# 04 · Heuristic quality filters (C4 and Gopher rules)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pretraining-data/04_quality_filters.ipynb) &nbsp; 📖 Concept deck: [Quality filtering](https://nilesh-salpe.github.io/visual-ai/pretraining-data/06-quality-filtering/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** the C4 line and page rules and the Gopher (MassiveText) document rules as functions, applied to a small set of sample documents with an explanation of why each one is kept or dropped.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import re
DOCS = {
    "article": "Gradient descent updates the weights in the direction that reduces the loss. The step size is set by the learning rate. If it is too large, training diverges. If it is too small, training is slow. Modern language models use the Adam optimizer with a warmup schedule. This keeps early updates small while statistics settle. Later the learning rate decays, letting the model settle into a good region of the loss surface.",
    "nav": "Home\nProducts\nAbout us\nContact\nLog in\nSign up",
    "spam": "best cheap shoes best cheap shoes best cheap shoes best cheap shoes best cheap shoes best cheap shoes best cheap shoes best cheap shoes best cheap shoes best cheap shoes",
    "code": "function add(a, b) { return a + b; } const x = add(1, 2); console.log(x); if (x > 2) { alert('big'); } // this is a comment line that goes on for a while to be long enough",
    "hashtags": " ".join(["#ai #ml #deep #learning #love #photo #instagood #follow #nofilter #tbt"] * 4) + " this post is about the best day of my life and I have to say that the light was perfect with friends",

    "lorem": "Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur.",
}
STOP = {"the", "be", "to", "of", "and", "that", "have", "with"}

## Task 1: C4 rules

From Raffel et al. (2020): keep only lines ending in terminal punctuation (`.`, `!`, `?`, or a closing quote) with at least 5 words; drop the page if fewer than 3 sentences remain, or if it contains "lorem ipsum" or a curly bracket `{`; drop lines containing "javascript".
Return the cleaned text, or `None` if the page is dropped. (Count sentences as terminal punctuation marks in the kept text.)

In [ ]:
def c4_clean(text):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_c4_clean(f):
    assert f(DOCS["article"]) is not None
    for k in ["nav", "code", "lorem", "spam"]:
        assert f(DOCS[k]) is None, f"C4 should drop '{k}'"
    assert f("Enable javascript to view this site. " * 1 + "This sentence has five words here. Another sentence also has words. A third sentence ends now.") is not None

check("c4_clean", _test_c4_clean, c4_clean)

## Task 2: Gopher document rules

From Rae et al. (2021): keep a document only if it has 50–100,000 words; mean word length 3–10 characters; symbol-to-word ratio ≤ 0.1 for `#` and for `...`/`…`; ≤ 90% of lines starting with a bullet; ≤ 30% of lines ending in an ellipsis; ≥ 80% of words containing a letter; and at least 2 of the stop words (the, be, to, of, and, that, have, with).
Return `(ok, reason)` with the first failing rule's name as reason (or `"ok"`).

In [ ]:
def gopher_ok(text):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_gopher_ok(f):
    assert f(DOCS["article"])[0] is True
    assert f(DOCS["nav"]) == (False, "word_count")
    assert f(DOCS["hashtags"]) == (False, "symbol_ratio")
    assert f(DOCS["lorem"])[1] == "stop_words", "Latin placeholder has no English stop words"

check("gopher_ok", _test_gopher_ok, gopher_ok)

## Task 3: Repetition: duplicate n-gram fraction

Gopher also removes documents whose characters are dominated by repeated n-grams. Return the fraction of words that belong to a word **3-gram that occurs more than once** in the document, and `ok = fraction ≤ threshold` (use 0.3 here as a simple stand-in for Gopher's per-n thresholds). Return `(ok, fraction)`.

In [ ]:
def repetition_ok(text, threshold=0.3):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_repetition_ok(f):
    ok, fr = f(DOCS["spam"]); assert not ok and fr > 0.9
    ok2, fr2 = f(DOCS["article"]); assert ok2 and fr2 < 0.2

check("repetition_ok", _test_repetition_ok, repetition_ok)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: C4 rules
def c4_clean_solution(text):
    low = text.lower()
    if "lorem ipsum" in low or "{" in text: return None
    lines = re.split(r"\n|(?<=[.!?\"”])\s+", text)
    keep = [l.strip() for l in lines if l.strip() and l.strip()[-1] in '.!?"”' and len(l.split()) >= 5 and "javascript" not in l.lower()]
    out = " ".join(keep)
    return out if len(re.findall(r'[.!?]["”]?(\s|$)', out)) >= 3 else None

check("c4_clean (reference)", _test_c4_clean, c4_clean_solution)

In [ ]:
# @title Solution for Task 2: Gopher document rules
def gopher_ok_solution(text):
    words = text.split(); n = len(words)
    if not 50 <= n <= 100_000: return False, "word_count"
    if not 3 <= sum(len(w) for w in words) / n <= 10: return False, "mean_word_length"
    if text.count("#") / n > 0.1 or (text.count("...") + text.count("…")) / n > 0.1: return False, "symbol_ratio"
    lines = [l for l in text.splitlines() if l.strip()] or [text]
    if sum(l.lstrip().startswith(("•", "-", "*")) for l in lines) / len(lines) > 0.9: return False, "bullets"
    if sum(l.rstrip().endswith(("...", "…")) for l in lines) / len(lines) > 0.3: return False, "ellipsis_lines"
    if sum(bool(re.search(r"[A-Za-z]", w)) for w in words) / n < 0.8: return False, "alphabetic"
    if len(STOP & {w.lower().strip(".,;:!?") for w in words}) < 2: return False, "stop_words"
    return True, "ok"

check("gopher_ok (reference)", _test_gopher_ok, gopher_ok_solution)

In [ ]:
# @title Solution for Task 3: Repetition: duplicate n-gram fraction
def repetition_ok_solution(text, threshold=0.3):
    w = text.lower().split(); grams = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    from collections import Counter
    c = Counter(grams); covered = set()
    for i, g in enumerate(grams):
        if c[g] > 1: covered.update(range(i, i + 3))
    fr = len(covered) / max(1, len(w))
    return fr <= threshold, fr

check("repetition_ok (reference)", _test_repetition_ok, repetition_ok_solution)

---
## Experiments
**Verdicts for every sample.** Which rules catch what? Different filters catch different junk, which is why pipelines stack them.

In [ ]:
for k, d in DOCS.items():
    c4 = c4_clean_solution(d) is not None; g, why = gopher_ok_solution(d); r, fr = repetition_ok_solution(d)
    print(f"{k:9s}: C4 {'keep' if c4 else 'drop'} | Gopher {'keep' if g else 'drop (' + why + ')'} | repetition {fr:.2f} {'ok' if r else 'drop'}")

---
## Takeaways
* Heuristic filters are cheap, transparent and catch most obvious junk: menus, code, placeholder text, spam.
* Each rule targets a specific failure; pipelines combine several (Dolma uses all Gopher rules plus C4's punctuation rule).
* Model-based filters (fastText or small transformers trained on "good" examples) catch subtler low quality; DCLM and FineWeb-Edu show they help most.